In [4]:
import sqlite3
import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool

def get_engine_for_chinook_db():
    """Pull sql file, populate in-memory database, and create engine."""
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)

    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )

engine = get_engine_for_chinook_db()
db = SQLDatabase(engine)

In [5]:
print(db.get_usable_table_names())

# Test Case 1
print(db.run("SELECT CustomerId, FirstName, LastName, Email, Phone FROM Customer WHERE Phone = '+55 (12) 3923-5555';"))
print(db.run("SELECT InvoiceId, InvoiceDate, Total FROM Invoice WHERE CustomerId = 1 ORDER BY InvoiceDate DESC LIMIT 1;"))

['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
[(1, 'Luís', 'Gonçalves', 'luisg@embraer.com.br', '+55 (12) 3923-5555')]
[(382, '2025-08-07 00:00:00', 8.91)]


In [6]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API key: ")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

print(llm.invoke("Say hi in 3 words").content)

Hey there, friend!


In [11]:
from langchain.agents import AgentState

class State(AgentState):
    """Represents the state of our LangGraph agent."""
    customer_id: str
    loaded_memory: str

In [12]:
from langchain_core.tools import tool

@tool
def get_albums_by_artist(artist: str):
    """Retrieves albums by a given artist (partial names allowed)."""
    return db.run(
        """
        SELECT Album.Title, Artist.Name
        FROM Album
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE :artist;
        """,
        parameters={"artist": f"%{artist}%"},
        include_columns=True,
    )

@tool
def get_tracks_by_artist(artist: str):
    """Retrieves tracks (songs) by a given artist or similar artists (partial names allowed)."""
    return db.run(
        """
        SELECT Track.Name AS SongName, Artist.Name AS ArtistName
        FROM Track
        JOIN Album ON Track.AlbumId = Album.AlbumId
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Artist.Name LIKE :artist
        LIMIT 20;
        """,
        parameters={"artist": f"%{artist}%"},
        include_columns=True,
    )

@tool
def get_songs_by_genre(genre: str):
    """Fetches songs that match a specific genre (e.g. Rock, Jazz, Latin)."""
    return db.run(
        """
        SELECT Track.Name AS SongName, Artist.Name AS ArtistName, Genre.Name AS Genre
        FROM Track
        JOIN Genre ON Track.GenreId = Genre.GenreId
        JOIN Album ON Track.AlbumId = Album.AlbumId
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Genre.Name LIKE :genre
        LIMIT 20;
        """,
        parameters={"genre": f"%{genre}%"},
        include_columns=True,
    )

@tool
def check_for_songs(song_title: str):
    """Checks if a song exists in the catalog by its name (partial names allowed)."""
    return db.run(
        """
        SELECT Track.Name AS SongName, Artist.Name AS ArtistName
        FROM Track
        JOIN Album ON Track.AlbumId = Album.AlbumId
        JOIN Artist ON Album.ArtistId = Artist.ArtistId
        WHERE Track.Name LIKE :song_title
        LIMIT 10;
        """,
        parameters={"song_title": f"%{song_title}%"},
        include_columns=True,
    )

music_tools = [get_albums_by_artist, get_tracks_by_artist, get_songs_by_genre, check_for_songs]

In [14]:
# test each tool
print(get_albums_by_artist.invoke({"artist": "Rolling Stones"}))
print(get_tracks_by_artist.invoke({"artist": "Rolling Stones"})[:300])
print(get_songs_by_genre.invoke({"genre": "Jazz"})[:300])
print(check_for_songs.invoke({"song_title": "Satisfaction"}))

[{'Title': 'Hot Rocks, 1964-1971 (Disc 1)', 'Name': 'The Rolling Stones'}, {'Title': 'No Security', 'Name': 'The Rolling Stones'}, {'Title': 'Voodoo Lounge', 'Name': 'The Rolling Stones'}]
[{'SongName': 'Time Is On My Side', 'ArtistName': 'The Rolling Stones'}, {'SongName': 'Heart Of Stone', 'ArtistName': 'The Rolling Stones'}, {'SongName': 'Play With Fire', 'ArtistName': 'The Rolling Stones'}, {'SongName': 'Satisfaction', 'ArtistName': 'The Rolling Stones'}, {'SongName': 'As Tears Go
[{'SongName': 'Desafinado', 'ArtistName': 'Antônio Carlos Jobim', 'Genre': 'Jazz'}, {'SongName': 'Garota De Ipanema', 'ArtistName': 'Antônio Carlos Jobim', 'Genre': 'Jazz'}, {'SongName': 'Samba De Uma Nota Só (One Note Samba)', 'ArtistName': 'Antônio Carlos Jobim', 'Genre': 'Jazz'}, {'SongName': 'Po
[{'SongName': 'Satisfaction', 'ArtistName': 'The Rolling Stones'}]


In [ ]:
# prompt = system message + list of history messages
from langchain_core.messages import SystemMessage

MUSIC_PROMPT = """You are the Music Catalog Assistant for a digital music store.
Your ONLY job is helping customers explore the catalog: albums, artists, tracks, and genres.

Tools:
- get_albums_by_artist: albums by an artist
- get_tracks_by_artist: songs by an artist or similar artists
- get_songs_by_genre: songs in a genre
- check_for_songs: whether a specific song exists

Rules:
1. Always use a tool to look things up. Never invent albums or songs that a tool did not return.
2. If a tool returns nothing, say so politely and suggest an alternative (similar artist or genre).
3. For recommendations, use the customer's saved preferences below.
4. Do NOT answer invoice or billing questions. Another agent handles those.
5. Keep answers short and formatted as a list.

Customer's saved music preferences: {memory}
"""

def music_prompt(state: State):
    memory = state.get("loaded_memory") or "None yet"
    system = SystemMessage(MUSIC_PROMPT.format(memory=memory))
    return [system] + state["messages"]